# Smart Police Patrol — Spatio-Temporal Violation Forecasting
**Weekly walk-forward pipeline (batch = 1 week of data)**

Forecasts next-24/48h parking-violation counts per H3 cell, then derives severity,
hotspot ranking and patrol allocation.

**Design notes baked in from data profiling**
- Data is **Nov 2023 → Apr 2024** (~150 days, 298K rows), Bengaluru.
- `violation_type` is a multi-label JSON array; WRONG + NO PARKING ≈ 83% of volume.
- Timestamps are **anonymized/shifted** (all seconds = `:46`) → model relative day/weekday
  patterns, do **not** read absolute clock hours as real enforcement times.
- **H3 res = 9** + **daily** grain (finer ~0.1 km² cells; MIN_CELL_EVENTS=50 to stay trainable).
- Count target → **Poisson** objective (not plain regression).
- **XGBoost** (`count:poisson`) replaces LightGBM; better regularisation on sparse res-9 data.
- **Weekly walk-forward**: each batch = one week; train on all prior weeks, predict the next.


## 0 · Setup & configuration

In [ ]:
import pandas as pd, numpy as np, json, h3, warnings, time
import xgboost as xgb
warnings.filterwarnings('ignore')

# ---- CONFIG: edit CSV_PATH to your file ----
CSV_PATH        = 'jan to may police violation_anonymized791b166.csv'
H3_RES          = 9            # 9 = finer ~0.1 km² cells (was 8)
TZ              = 'Asia/Kolkata'
MIN_CELL_EVENTS = 50           # lowered from 100; res-9 cells see fewer events per cell
MIN_TRAIN_WEEKS = 3            # warm-up weeks before walk-forward starts
TRAIN_WINDOW_WEEKS = None      # None = expanding window; int = sliding (last N weeks)
RANDOM_STATE    = 42


## 1 · Load & clean
Drop the 100%-null columns, parse timestamps, geo-filter to Bengaluru, assign H3 cells.

In [ ]:
t0 = time.time()
df = pd.read_csv(CSV_PATH, low_memory=False, usecols=[
    'id','latitude','longitude','created_datetime',
    'police_station','violation_type','vehicle_type'])

df['dt'] = pd.to_datetime(df['created_datetime'], errors='coerce', utc=True).dt.tz_convert(TZ)
df = df.dropna(subset=['dt','latitude','longitude'])
df = df[df.latitude.between(12.7, 13.2) & df.longitude.between(77.3, 77.9)]   # drop geo outliers
df['date'] = df['dt'].dt.floor('D').dt.tz_localize(None)                       # IST calendar date
df['cell'] = [h3.latlng_to_cell(la, lo, H3_RES) for la, lo in zip(df.latitude, df.longitude)]

print(f'rows={len(df):,}  cells={df.cell.nunique()}  '
      f'span={df.date.min().date()}..{df.date.max().date()}  ({round(time.time()-t0,1)}s)')


## 2 · Daily panel + zero-fill + weekly batches
Aggregate to one row per **(cell, day)**, then **zero-fill** the full calendar grid so the
model sees the days a cell had *no* violations (forecasting needs explicit zeros).
Only cells with `>= MIN_CELL_EVENTS` are modeled; sparser cells fall back to a baseline.
`week_idx` defines the **weekly batch** used for walk-forward training.

In [ ]:
# daily counts for ALL cells (needed for neighbour features)
daily = df.groupby(['cell','date']).size().rename('count').reset_index()

# qualifying (modeled) cells
vc = df.cell.value_counts()
keep = vc[vc >= MIN_CELL_EVENTS].index
print(f'modeled cells: {len(keep)} of {df.cell.nunique()}')

# zero-filled panel over the full date range, qualifying cells only
dates = pd.date_range(df.date.min(), df.date.max(), freq='D')
idx   = pd.MultiIndex.from_product([keep, dates], names=['cell','date'])
panel = (daily.set_index(['cell','date']).reindex(idx, fill_value=0).reset_index())

# attach the dominant station per cell (categorical)
stmap = df.groupby('cell')['police_station'].agg(lambda s: s.mode().iloc[0])
panel['station'] = panel['cell'].map(stmap).astype('category')

panel = panel.sort_values(['cell','date']).reset_index(drop=True)

# WEEKLY BATCH INDEX  ->  each week = one batch (7 day-rows per cell)
panel['week_idx'] = ((panel.date - panel.date.min()).dt.days // 7).astype(int)
print('panel shape', panel.shape, '| weeks', panel.week_idx.nunique())


## 3 · Targets
`y_t1` = next-day count (24h horizon), `y_t2` = day after, `y_48h` = their sum (drives severity).

In [ ]:
g = panel.groupby('cell')['count']
panel['y_t1']  = g.shift(-1)
panel['y_t2']  = g.shift(-2)
panel['y_48h'] = panel['y_t1'] + panel['y_t2']


## 4 · Feature engineering
All history features use `.shift(1)` (or `shift(L>=1)`) so they only see days **before** the
target — the panel is sorted by `cell, date` so groupby-shift is leakage-safe.

**4.1 Calendar** (known for any future date)

In [ ]:
import holidays
ka  = holidays.India(subdiv='KA', years=[2023, 2024])
hol = pd.to_datetime([d for d in ka if panel.date.min() <= pd.Timestamp(d) <= panel.date.max()])

panel['dow']            = panel.date.dt.dayofweek
panel['is_weekend']     = (panel.dow >= 5).astype(int)
panel['month']          = panel.date.dt.month
panel['day_of_month']   = panel.date.dt.day
panel['week_of_period'] = panel.week_idx
panel['dow_sin']        = np.sin(2*np.pi*panel.dow/7)
panel['dow_cos']        = np.cos(2*np.pi*panel.dow/7)
panel['is_holiday']     = panel.date.isin(hol).astype(int)
panel['days_from_holiday'] = panel.date.apply(lambda d:(d-hol[hol<=d]).days.min() if (hol<=d).any() else 999)
panel['days_to_holiday']   = panel.date.apply(lambda d:(hol[hol>=d]-d).days.min() if (hol>=d).any() else 999)


**4.2 Lags, rolling stats, trend** (all shifted)

In [ ]:
g = panel.groupby('cell')['count']
for L in [1,2,3,7,14]:
    panel[f'lag_{L}'] = g.shift(L)

def roll(s, w, fn):                       # window ends at t-1 (shift before rolling)
    return s.shift(1).rolling(w, min_periods=max(2, w//2)).agg(fn)

gp = panel.groupby('cell')['count']
panel['roll_mean_3']  = gp.transform(lambda s: roll(s, 3,  'mean'))
panel['roll_mean_7']  = gp.transform(lambda s: roll(s, 7,  'mean'))
panel['roll_mean_14'] = gp.transform(lambda s: roll(s, 14, 'mean'))
panel['roll_std_7']   = gp.transform(lambda s: roll(s, 7,  'std'))
panel['roll_max_7']   = gp.transform(lambda s: roll(s, 7,  'max'))
panel['roll_median_7']= gp.transform(lambda s: roll(s, 7,  'median'))
panel['ewm_mean_7']   = gp.transform(lambda s: s.shift(1).ewm(span=7, min_periods=2).mean())

# trend / acceleration
panel['accel']  = panel.roll_mean_3 - panel.roll_mean_7     # short vs medium level
panel['diff_1'] = panel.lag_1 - panel.lag_2
prevwk = gp.transform(lambda s: roll(s, 7, 'mean').shift(7))
panel['growth_7'] = (panel.roll_mean_7 - prevwk) / (prevwk + 1)


**4.3 Same-weekday & cell expanding means** (leakage-safe via `.expanding()`)

In [ ]:
def past_dow_mean(d):
    d = d.sort_values('date')
    return d.groupby('dow')['count'].apply(lambda s: s.shift(1).expanding().mean())

panel['same_dow_mean'] = (panel.groupby('cell', group_keys=False)
                               .apply(past_dow_mean).reset_index(level=0, drop=True))
panel['cell_expanding_mean'] = gp.transform(lambda s: s.shift(1).expanding().mean())
panel['dev_from_dow'] = panel.lag_7 - panel.same_dow_mean


**4.4 Spatial — H3 neighbour ring** (yesterday's counts in the 6 adjacent cells)

In [ ]:
nbrs = {c: [x for x in h3.grid_disk(c, 1) if x != c] for c in keep}
wide = panel.pivot_table(index='date', columns='cell', values='count', fill_value=0)
prev = wide.shift(1)                              # yesterday
ns = {}
for c, nn in nbrs.items():
    cols = [x for x in nn if x in prev.columns]
    ns[c] = prev[cols].sum(axis=1) if cols else pd.Series(0, index=prev.index)
nbr_df = pd.DataFrame(ns).stack().rename('nbr_count_lag1').reset_index()
nbr_df.columns = ['date','cell','nbr_count_lag1']
panel = panel.merge(nbr_df, on=['date','cell'], how='left')


**4.5 Composition shares** — slow-moving cell 'personality' (lagged rolling shares, leakage-safe)

In [ ]:
BUCKET = {'WRONG PARKING':'wrong','NO PARKING':'no_parking',
          'PARKING IN A MAIN ROAD':'road','PARKING NEAR ROAD CROSSING':'road',
          'PARKING NEAR TRAFFIC LIGHT OR ZEBRA CROSS':'road','PARKING ON FOOTPATH':'footpath'}
def primary(x):
    try: t = json.loads(x)
    except: return 'other'
    return BUCKET.get(t[0], 'other') if t else 'other'
df['vbucket'] = df['violation_type'].apply(primary)
df['is2w']    = df['vehicle_type'].isin(['SCOOTER','MOTOR CYCLE','MOPED'])

shares = df.groupby(['cell','date']).agg(
    share_wrong   =('vbucket', lambda s:(s=='wrong').mean()),
    share_nopark  =('vbucket', lambda s:(s=='no_parking').mean()),
    share_2wheeler=('is2w','mean')).reset_index()
panel = panel.merge(shares, on=['cell','date'], how='left')

# lag + 7d rolling so a row never uses its own day's composition
for col in ['share_wrong','share_nopark','share_2wheeler']:
    panel[col] = (panel.groupby('cell')[col]
                       .transform(lambda s: s.shift(1).rolling(7, min_periods=1).mean()))

print('features built — total cols', panel.shape[1], f'({round(time.time()-t0,1)}s)')


## 5 · Weekly walk-forward training — **batch = 1 week**
For each week `w` (after the warm-up), train on **all prior weeks** (expanding) or the last
`TRAIN_WINDOW_WEEKS` (sliding) and predict the **single week `w`**. Every test batch holds
exactly one week of data. XGBoost with a **Poisson** objective (`count:poisson`). `enable_categorical=True` handles the station column directly.

In [ ]:
FEATS = ['dow','is_weekend','month','day_of_month','week_of_period','dow_sin','dow_cos',
 'is_holiday','days_to_holiday','days_from_holiday',
 'lag_1','lag_2','lag_3','lag_7','lag_14',
 'roll_mean_3','roll_mean_7','roll_mean_14','roll_std_7','roll_max_7','roll_median_7','ewm_mean_7',
 'accel','diff_1','growth_7','same_dow_mean','cell_expanding_mean','dev_from_dow',
 'nbr_count_lag1','share_wrong','share_nopark','share_2wheeler','station']

def xgb_model():
    return xgb.XGBRegressor(
        objective='count:poisson',
        n_estimators=600,
        learning_rate=0.03,
        max_depth=6,
        min_child_weight=5,    # min sum-of-weights per leaf
        gamma=0.1,             # min loss-reduction required for a split
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        tree_method='hist',    # required for enable_categorical
        enable_categorical=True,
        random_state=RANDOM_STATE,
        verbosity=0,
    )

data  = panel.dropna(subset=['y_t1']).copy()
data['station'] = data['station'].astype('category')   # ensure category dtype for XGBoost

weeks = sorted(data.week_idx.unique())
preds = []

for w in weeks:
    if w < MIN_TRAIN_WEEKS:
        continue
    if TRAIN_WINDOW_WEEKS is None:
        tr = data[data.week_idx < w].copy()                                   # expanding
    else:
        tr = data[(data.week_idx < w) & (data.week_idx >= w-TRAIN_WINDOW_WEEKS)].copy()
    te = data[data.week_idx == w].copy()                                      # ONE WEEK BATCH
    if len(te) == 0:
        continue

    # unify category levels so XGBoost sees the same schema in train and test
    all_cats = tr['station'].cat.categories.union(te['station'].cat.categories)
    tr['station'] = tr['station'].cat.set_categories(all_cats)
    te['station'] = te['station'].cat.set_categories(all_cats)

    m = xgb_model()
    m.fit(tr[FEATS], tr['y_t1'])
    out = te[['cell','date','week_idx','y_t1','lag_7','cell_expanding_mean']].copy()
    out['pred']  = np.clip(m.predict(te[FEATS]), 0, None)
    out['naive'] = out['lag_7'].fillna(out['cell_expanding_mean']).fillna(0)
    preds.append(out)

preds = pd.concat(preds, ignore_index=True)
print(f'walk-forward done: {preds.week_idx.nunique()} weekly batches, {len(preds):,} predictions')

# capture the final trained model (trained on all weeks except last) for persistence
last_model = m
last_feats  = FEATS
last_cats   = all_cats   # station category levels seen during last training fold


## 6 · Evaluation — MAE / RMSE / Poisson deviance / Precision@K

In [ ]:
def poisson_dev(y, p):
    p = np.clip(p, 1e-6, None)
    return 2*np.mean(np.where(y>0, y*np.log(y/p), 0) - (y-p))

def precision_at_k(frame, k=10):
    # per day: of the k cells we ranked highest, how many were in the true top-k
    hits = []
    for _, gdf in frame.groupby('date'):
        if len(gdf) < k: continue
        true_top = set(gdf.nlargest(k, 'y_t1').cell)
        pred_top = set(gdf.nlargest(k, 'pred').cell)
        hits.append(len(true_top & pred_top)/k)
    return np.mean(hits)

y, p, nv = preds.y_t1.values, preds.pred.values, preds.naive.values
print('            model     naive')
print(f'MAE      {np.abs(p-y).mean():8.3f}  {np.abs(nv-y).mean():8.3f}')
print(f'RMSE     {np.sqrt(((p-y)**2).mean()):8.3f}  {np.sqrt(((nv-y)**2).mean()):8.3f}')
print(f'PoisDev  {poisson_dev(y,p):8.3f}  {poisson_dev(y,nv):8.3f}')
print(f'P@10     {precision_at_k(preds,10):8.3f}  '
      f'{precision_at_k(preds.assign(pred=preds.naive),10):8.3f}')
print(f'P@25     {precision_at_k(preds,25):8.3f}  '
      f'{precision_at_k(preds.assign(pred=preds.naive),25):8.3f}')

# per-week MAE table
wk = (preds.assign(ae=np.abs(preds.pred-preds.y_t1),
                   ae_n=np.abs(preds.naive-preds.y_t1))
            .groupby('week_idx').agg(n=('y_t1','size'),
                                     mae_model=('ae','mean'),
                                     mae_naive=('ae_n','mean')).round(3))
wk


## 7 · (Optional) XGBoost + LightGBM + CatBoost ensemble
Same weekly walk-forward, averaged across three boosters. Set `USE_ENSEMBLE=True`.
XGBoost is primary. LightGBM/CatBoost optional; falls back silently if not installed.

In [ ]:
USE_ENSEMBLE = False

if USE_ENSEMBLE:
    models = {'xgb': xgb_model}   # XGBoost is the primary model
    try:
        import lightgbm as lgb
        models['lgb'] = lambda: lgb.LGBMRegressor(objective='poisson', n_estimators=600,
            learning_rate=0.03, num_leaves=31, min_child_samples=20,
            subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE, verbose=-1)
    except Exception as e: print('lightgbm unavailable:', e)
    try:
        from catboost import CatBoostRegressor
        models['cat'] = lambda: CatBoostRegressor(loss_function='Poisson', iterations=600,
            learning_rate=0.03, depth=6, cat_features=['station'],
            random_state=RANDOM_STATE, verbose=0)
    except Exception as e: print('catboost unavailable:', e)

    ens = []
    for w in weeks:
        if w < MIN_TRAIN_WEEKS: continue
        tr = data[data.week_idx < w].copy() if TRAIN_WINDOW_WEEKS is None              else data[(data.week_idx < w) & (data.week_idx >= w-TRAIN_WINDOW_WEEKS)].copy()
        te = data[data.week_idx == w].copy()
        if len(te) == 0: continue
        all_cats = tr['station'].cat.categories.union(te['station'].cat.categories)
        tr['station'] = tr['station'].cat.set_categories(all_cats)
        te['station'] = te['station'].cat.set_categories(all_cats)
        ps = []
        for name, mk in models.items():
            m = mk()
            if name == 'cat':
                m.fit(tr[FEATS], tr['y_t1'])
            elif name == 'lgb':
                m.fit(tr[FEATS], tr['y_t1'], categorical_feature=['station'])
            else:
                m.fit(tr[FEATS], tr['y_t1'])
            ps.append(np.clip(m.predict(te[FEATS]), 0, None))
        o = te[['cell','date','y_t1']].copy()
        o['pred'] = np.mean(ps, axis=0)
        ens.append(o)
    ens = pd.concat(ens, ignore_index=True)
    ye, pe = ens.y_t1.values, ens.pred.values
    print(f'ENSEMBLE  MAE {np.abs(pe-ye).mean():.3f}  RMSE {np.sqrt(((pe-ye)**2).mean()):.3f}')
else:
    print('USE_ENSEMBLE = False  (XGBoost-only results above)')


## 8 · Severity engine + patrol allocation
Severity thresholds are **calibrated to the realized 48h-count percentiles** (not guessed),
then mapped to officer counts. Risk score drives hotspot ranking.

In [ ]:
# use the most recent modeled week's predictions as the 'operational' forecast
latest = preds[preds.week_idx == preds.week_idx.max()].copy()

# calibrate severity cutoffs from realized distribution
q = preds.y_t1.quantile([0.50, 0.80, 0.95]).values
def severity(c):
    if c <= q[0]: return 'Low'
    if c <= q[1]: return 'Medium'
    if c <= q[2]: return 'High'
    return 'Critical'
OFFICERS = {'Low':1, 'Medium':2, 'High':3, 'Critical':4}

latest['severity'] = latest['pred'].apply(severity)
latest['officers'] = latest['severity'].map(OFFICERS)
# risk score = level x recent growth x neighbourhood pressure
latest = latest.merge(panel[['cell','date','growth_7','nbr_count_lag1']], on=['cell','date'], how='left')
latest['risk_score'] = (latest['pred']
                        * (1 + latest['growth_7'].fillna(0).clip(-0.9, None))
                        * (1 + latest['nbr_count_lag1'].fillna(0)/ (latest['nbr_count_lag1'].mean()+1)))

print('severity cutoffs (p50/p80/p95):', q.round(2))
print('\nTop 10 critical hotspots for', latest.date.max().date())
top = latest.sort_values('risk_score', ascending=False).head(10)
top[['cell','pred','severity','officers','risk_score']].round(2)


## 9 · Save outputs

In [ ]:
out_cols = ['cell','date','pred','severity','officers','risk_score']
latest_sorted = latest.sort_values('risk_score', ascending=False)[out_cols].round(3)
latest_sorted.to_csv('patrol_forecast_latest.csv', index=False)
try:
    preds.to_parquet('walkforward_predictions.parquet', index=False)   # needs pyarrow
    print('saved: patrol_forecast_latest.csv , walkforward_predictions.parquet')
except Exception:
    preds.to_csv('walkforward_predictions.csv', index=False)           # CSV fallback
    print('saved: patrol_forecast_latest.csv , walkforward_predictions.csv')
latest_sorted.head(25)

# ── Persist XGBoost model + metadata ──────────────────────────────────────────
import joblib, os
os.makedirs('model_artifacts', exist_ok=True)
last_model.save_model('model_artifacts/xgb_patrol_model.json')     # XGBoost native format
joblib.dump({'feats': last_feats, 'station_cats': last_cats,
             'h3_res': H3_RES, 'severity_quantiles': q.tolist()},
            'model_artifacts/model_meta.pkl')
print('model saved -> model_artifacts/xgb_patrol_model.json + model_meta.pkl')


## 10 · Map the hotspots on Bengaluru (Leaflet)
Draws each H3 cell as its real hexagon polygon, coloured by predicted severity, with a
popup (predicted count / officers / risk). Saves a standalone `patrol_map.html` and renders
it inline.

In [ ]:
def build_leaflet_map(dfm, out_html='patrol_map.html', center=(12.9716, 77.5946), zoom=11):
    """dfm needs one row per cell: cell, pred, severity, officers, risk_score."""
    COLORS = {'Low':'#2ecc71','Medium':'#f1c40f','High':'#e67e22','Critical':'#e74c3c'}
    feats = []
    for _, r in dfm.iterrows():
        b = h3.cell_to_boundary(r['cell'])                       # [(lat,lng), ...]
        ring = [[lng, lat] for lat, lng in b] + [[b[0][1], b[0][0]]]
        feats.append({"type":"Feature",
            "properties":{"cell":r['cell'],"pred":round(float(r['pred']),1),
                "severity":r['severity'],"officers":int(r['officers']),
                "risk":round(float(r['risk_score']),1),"color":COLORS.get(r['severity'],'#888')},
            "geometry":{"type":"Polygon","coordinates":[ring]}})
    gj = {"type":"FeatureCollection","features":feats}
    counts = dfm['severity'].value_counts().to_dict()
    legend = "".join(f'<div><span style="background:{c}"></span>{s} ({counts.get(s,0)})</div>'
                     for s, c in COLORS.items())
    html = f"""<!DOCTYPE html><html><head><meta charset="utf-8">
<title>Patrol Forecast - Bengaluru</title>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"/>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<style>
 html,body,#map{{height:100%;margin:0}}
 .legend{{position:absolute;bottom:20px;right:12px;z-index:1000;background:#fff;
   padding:10px 12px;border-radius:8px;font:13px system-ui;box-shadow:0 1px 6px rgba(0,0,0,.3)}}
 .legend div{{display:flex;align-items:center;gap:6px;margin:2px 0}}
 .legend span{{width:14px;height:14px;border-radius:3px;display:inline-block}}
 .legend b{{display:block;margin-bottom:4px}}
</style></head><body>
<div id="map"></div>
<div class="legend"><b>Predicted severity</b>{legend}</div>
<script>
const data={json.dumps(gj)};
const map=L.map('map').setView([{center[0]},{center[1]}],{zoom});
L.tileLayer('https://{{s}}.tile.openstreetmap.org/{{z}}/{{x}}/{{y}}.png',
  {{maxZoom:19,attribution:'(c) OpenStreetMap'}}).addTo(map);
L.geoJSON(data,{{
  style:f=>({{color:'#333',weight:1,fillColor:f.properties.color,fillOpacity:0.55}}),
  onEachFeature:(f,l)=>{{const p=f.properties;
    l.bindPopup(`<b>${{p.severity}}</b><br>Predicted: ${{p.pred}}<br>`+
      `Officers: ${{p.officers}}<br>Risk: ${{p.risk}}<br><code>${{p.cell}}</code>`);
    l.bindTooltip(`${{p.pred}}`,{{direction:'center'}});}}
}}).addTo(map);
</script></body></html>"""
    with open(out_html, 'w') as f:
        f.write(html)
    return out_html

# snapshot the single most recent forecast date, one row per cell
snap = latest[latest.date == latest.date.max()].drop_duplicates('cell')
build_leaflet_map(snap, 'patrol_map.html')
print(f'mapped {len(snap)} cells for {snap.date.max().date()} -> patrol_map.html')

from IPython.display import IFrame
IFrame('patrol_map.html', width='100%', height=560)


## 11 · Validate on unseen test data — Predicted vs Actual map
Aggregates the **final `TEST_WEEKS` weeks the model never trained on** (each forecast is
out-of-sample from the walk-forward). Two toggleable Leaflet layers share one colour scale,
so **matching colours between Predicted and Actual = a good forecast**. Header shows MAE and
Precision@10 on this unseen window.

In [ ]:
TEST_WEEKS = 4

def build_test_map(preds, test_weeks=TEST_WEEKS, out_html='patrol_map_test.html',
                   center=(12.9716, 77.5946), zoom=11):
    wmax = preds.week_idx.max()
    test = preds[preds.week_idx >= wmax - test_weeks + 1].copy()
    d0, d1 = pd.to_datetime(test.date).min(), pd.to_datetime(test.date).max()
    agg = test.groupby('cell').agg(pred=('pred','sum'), actual=('y_t1','sum')).reset_index()
    agg['err'] = agg.pred - agg.actual

    mae = np.abs(test.pred - test.y_t1).mean()
    def p_at_k(frame, k=10):
        hits=[]
        for _, g in frame.groupby('date'):
            if len(g) < k: continue
            hits.append(len(set(g.nlargest(k,'y_t1').cell) & set(g.nlargest(k,'pred').cell))/k)
        return np.mean(hits)
    p10 = p_at_k(test, 10)

    qs = np.quantile(agg.actual, [0,.2,.4,.6,.8,1.0])           # shared scale from actuals
    RAMP = ['#fee5d9','#fcae91','#fb6a4a','#de2d26','#a50f15']
    def color(v):
        for i in range(5):
            if v <= qs[i+1]: return RAMP[i]
        return RAMP[-1]
    def layer(valcol):
        feats=[]
        for _, r in agg.iterrows():
            b = h3.cell_to_boundary(r['cell'])
            ring = [[lng,lat] for lat,lng in b] + [[b[0][1], b[0][0]]]
            feats.append({"type":"Feature","properties":{
                "cell":r['cell'],"pred":round(float(r.pred),1),"actual":round(float(r.actual),1),
                "err":round(float(r.err),1),"color":color(r[valcol])},
                "geometry":{"type":"Polygon","coordinates":[ring]}})
        return {"type":"FeatureCollection","features":feats}
    gj_pred, gj_act = layer('pred'), layer('actual')
    legend = "".join(f'<div><span style="background:{c}"></span>{int(qs[i])}-{int(qs[i+1])}</div>'
                     for i, c in enumerate(RAMP))

    html = f"""<!DOCTYPE html><html><head><meta charset="utf-8">
<title>Unseen Test</title>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"/>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<style>
 html,body,#map{{height:100%;margin:0}}
 .panel{{position:absolute;top:10px;left:50px;z-index:1000;background:#fff;padding:8px 12px;
   border-radius:8px;font:13px system-ui;box-shadow:0 1px 6px rgba(0,0,0,.3);max-width:330px}}
 .legend{{position:absolute;bottom:20px;right:12px;z-index:1000;background:#fff;padding:10px 12px;
   border-radius:8px;font:12px system-ui;box-shadow:0 1px 6px rgba(0,0,0,.3)}}
 .legend div{{display:flex;align-items:center;gap:6px;margin:2px 0}}
 .legend span{{width:14px;height:14px;border-radius:3px;display:inline-block}}
</style></head><body>
<div id="map"></div>
<div class="panel"><b>Unseen test window</b><br>{{d0}} -> {{d1}} ({test_weeks} weeks, never trained on)<br>
 MAE {mae:.2f} · Precision@10 {p10:.0%}<br>
 <small>Toggle Predicted/Actual (top-right). Same scale -> matching colours = good forecast.</small></div>
<div class="legend"><b>Violations / window</b>{legend}</div>
<script>
const map=L.map('map').setView([{center[0]},{center[1]}],{zoom});
L.tileLayer('https://{{s}}.tile.openstreetmap.org/{{z}}/{{x}}/{{y}}.png',
  {{maxZoom:19,attribution:'(c) OpenStreetMap'}}).addTo(map);
function mk(data){{return L.geoJSON(data,{{
  style:f=>({{color:'#333',weight:1,fillColor:f.properties.color,fillOpacity:0.6}}),
  onEachFeature:(f,l)=>{{const p=f.properties;
    l.bindPopup(`<b>${{p.cell}}</b><br>Predicted: ${{p.pred}}<br>Actual: ${{p.actual}}<br>Error: ${{p.err}}`);}}
}});}}
const predL=mk({json.dumps(gj_pred)}), actL=mk({json.dumps(gj_act)});
actL.addTo(map);
L.control.layers(null,{{"Actual (truth)":actL,"Predicted (model)":predL}},{{collapsed:false}}).addTo(map);
</script></body></html>""".replace('{{d0}}', str(d0.date())).replace('{{d1}}', str(d1.date()))
    with open(out_html,'w') as f: f.write(html)
    print(f'unseen test: {d0.date()}..{d1.date()} | {len(agg)} cells | MAE {mae:.2f} | P@10 {p10:.0%} -> {out_html}')
    return out_html

build_test_map(preds, TEST_WEEKS, 'patrol_map_test.html')
from IPython.display import IFrame
IFrame('patrol_map_test.html', width='100%', height=560)


---
### Notes for going further
- **Peak-hour window** per cell = argmax of its normalized 24-bin hour histogram (build from the
  raw events). Remember the clock is anonymized — treat as a *relative* pattern.
- **Dominant type** per cell ≈ historical mode of `vbucket` (near-deterministic given dominance).
- Tune via `xgb.cv` or optuna *inside* each training window only; never touch future weeks.
- Try `objective='tweedie'`, `tweedie_variance_power≈1.2` for the zero-inflated cells.
- Prune features with `xgb.plot_importance(m)` / `m.feature_importances_` after the first full run — ~150 days favours fewer strong features.
